In [5]:
# Nếu thiếu thư viện, chạy dòng này một lần:
%pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
import hashlib
import json
import re

from pathlib import Path
from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedGroupKFold
)

SEED = 42
INPUT_FILE = "Mathernal_Risk.csv"   
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = "RiskLevel"

BASIC_FEATURES = [
    "Age", "BodyTemp", "HeartRate",
    "SystolicBP", "DiastolicBP", "BMI"
]

ADDITIONAL_FEATURES = ["HbA1c", "FastingGlucose"]

FEATURES = BASIC_FEATURES + ADDITIONAL_FEATURES
INPUT_FILE = "Mathernal_Risk.csv"

print("Setup completed!")

Note: you may need to restart the kernel to use updated packages.
Setup completed!


In [6]:

# CELL 2 — LOAD & STANDARDIZE REAL DATASET

import pandas as pd
import numpy as np
import hashlib
import re

# 1. Read original dataset
raw = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")

print("Original shape:", raw.shape)
print("Original columns:")
print(raw.columns.tolist())

# 2. Rename columns based on the ACTUAL CSV
COLUMN_MAP = {
    "Patient ID": "Patient_ID",
    "Name": "Name",
    "Age": "Age",
    "Body Temperature(F) ": "BodyTemp",
    "Heart rate(bpm)": "HeartRate",
    "Systolic Blood Pressure(mm Hg)": "SystolicBP",
    "Diastolic Blood Pressure(mm Hg)": "DiastolicBP",
    "BMI(kg/m 2)": "BMI",
    "Blood Glucose(HbA1c)": "HbA1c",
    "Blood Glucose(Fasting hour-mg/dl)": "FastingGlucose",
    "Status": "RiskLevel"
}

# Strip trailing spaces before renaming
raw.columns = raw.columns.str.strip()
COLUMN_MAP = {
    key.strip(): value
    for key, value in COLUMN_MAP.items()
}

raw = raw.rename(columns=COLUMN_MAP)

# 3. Validate required columns
required_columns = FEATURES + [TARGET]

missing_cols = [
    col for col in required_columns
    if col not in raw.columns
]

assert not missing_cols, f"Missing columns: {missing_cols}"
assert len(raw) == 6103, f"Expected 6103 rows, got {len(raw)}"

# 4. Convert all predictors to numeric
for col in FEATURES:
    raw[col] = pd.to_numeric(
        raw[col], errors="coerce"
    )

assert raw[FEATURES].notna().all().all(), (
    "Missing or non-numeric values detected!"
)

# 5. Standardize target labels
def normalize_risk(value):
    key = re.sub(
        r"[\s_-]", "",
        str(value).strip().lower()
    )
    mapping = {
        "low": "Low",
        "lowrisk": "Low",
        "mid": "Mid",
        "midrisk": "Mid",
        "high": "High",
        "highrisk": "High"
    }
    return mapping.get(key)

raw[TARGET] = raw[TARGET].map(normalize_risk)

assert raw[TARGET].notna().all(), (
    "Unrecognized RiskLevel labels!"
)

# 6. Track original CSV rows
raw["source_row"] = np.arange(2, len(raw) + 2)

# 7. Calculate source file checksum
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

source_hash = sha256_file(INPUT_FILE)

# 8. Display results
print("\nSTANDARDIZATION SUCCESSFUL!")
print("Dataset shape:", raw.shape)
print("Features:", FEATURES)

print("\nRisk distribution:")
print(raw[TARGET].value_counts())

display(raw.head())


Original shape: (6103, 11)
Original columns:
['Patient ID', 'Name', 'Age', 'Body Temperature(F) ', 'Heart rate(bpm)', 'Systolic Blood Pressure(mm Hg)', 'Diastolic Blood Pressure(mm Hg)', 'BMI(kg/m 2)', 'Blood Glucose(HbA1c)', 'Blood Glucose(Fasting hour-mg/dl)', 'Status']

STANDARDIZATION SUCCESSFUL!
Dataset shape: (6103, 12)
Features: ['Age', 'BodyTemp', 'HeartRate', 'SystolicBP', 'DiastolicBP', 'BMI', 'HbA1c', 'FastingGlucose']

Risk distribution:
RiskLevel
High    2059
Mid     2043
Low     2001
Name: count, dtype: int64


,Patient_ID,Name,Age,BodyTemp,HeartRate,SystolicBP,DiastolicBP,BMI,HbA1c,FastingGlucose,RiskLevel,source_row
0,101,Bhavana,20,97.5,91,161,100,24.9,41,5.8,High,2
1,102,Mithila,45,97.7,99,99,94,22.1,36,5.7,High,3
2,103,Phoolan,29,98.6,84,129,87,19.0,42,6.4,Mid,4
3,104,Joyonti,26,99.5,135,161,101,23.7,46,4.5,High,5
4,105,Vana,38,102.5,51,106,91,18.8,38,4.3,High,6


In [7]:
# Kiểm tra đơn vị nhiệt độ trước khi cleaning
print("BodyTemp summary:")
print(raw["BodyTemp"].describe())

# Quy tắc loại dữ liệu
bad_age = raw["Age"] > 100

bad_temp = (
    (raw["BodyTemp"] < 95) |
    (raw["BodyTemp"] > 105)
)

bad_dbp = raw["DiastolicBP"] < 50

# Tổng hợp nguyên nhân
flags = pd.DataFrame({
    "Age > 100": bad_age,
    "Temperature outside 95-105 F": bad_temp,
    "DiastolicBP < 50": bad_dbp
}, index=raw.index)

remove_mask = flags.any(axis=1)

removed = raw.loc[remove_mask].copy()

removed["reason"] = flags.loc[remove_mask].apply(
    lambda row: "; ".join(row.index[row]),
    axis=1
)

cleaned = raw.loc[~remove_mask].copy()

# Cleaning report
cleaning_report = pd.DataFrame({
    "Check": [
        "Original records",
        "Age > 100",
        "Temperature outside 95-105 F",
        "DiastolicBP < 50",
        "Overlapping violations",
        "Total removed",
        "Remaining records"
    ],
    "Count": [
        len(raw),
        int(bad_age.sum()),
        int(bad_temp.sum()),
        int(bad_dbp.sum()),
        int((flags.sum(axis=1) > 1).sum()),
        len(removed),
        len(cleaned)
    ]
})

display(cleaning_report)

# Xuất các dòng bị loại để audit
removed.to_csv(
    OUTPUT_DIR / "removed_outliers_audit.csv",
    index=False
)

# Chỉ tiếp tục khi đúng cleaning protocol dự kiến
assert int(bad_age.sum()) == 1, "Age anomalies không khớp!"
assert int(bad_temp.sum()) == 43, "Temperature anomalies không khớp!"
assert int(bad_dbp.sum()) == 1, "DiastolicBP anomalies không khớp!"
assert int((flags.sum(axis=1) > 1).sum()) == 0, (
    "Có dòng vi phạm nhiều tiêu chí; cần kiểm tra."
)

assert len(cleaned) == 6058, (
    f"Sau cleaning còn {len(cleaned)} dòng, không phải 6058."
)

print("CLEANING PASSED!")
print("Cleaned records:", len(cleaned))
print(cleaned[TARGET].value_counts())

BodyTemp summary:
count    6103.000000
mean       98.665574
std         1.590983
min        39.600000
25%        98.600000
50%        98.600000
75%        98.800000
max       104.000000
Name: BodyTemp, dtype: float64


,Check,Count
0,Original records,6103
1,Age > 100,1
2,Temperature outside 95-105 F,43
3,DiastolicBP < 50,1
4,Overlapping violations,0
5,Total removed,45
6,Remaining records,6058


CLEANING PASSED!
Cleaned records: 6058
RiskLevel
Mid     2043
High    2016
Low     1999
Name: count, dtype: int64


In [8]:
# Sắp xếp ổn định trước khi gắn row_id
sort_cols = FEATURES + [TARGET]

df = cleaned.sort_values(
    sort_cols,
    kind="mergesort"
).reset_index(drop=True).copy()

df["row_id"] = [
    f"R{i:06d}"
    for i in range(1, len(df) + 1)
]

# Tạo định danh hồ sơ dựa trên 8 predictors
profile_key = df[FEATURES].astype(str).agg("|".join, axis=1)

n_duplicate_profiles = int(profile_key.duplicated().sum())

n_conflicting_profiles = int(
    df.groupby(profile_key)[TARGET]
      .nunique()
      .gt(1)
      .sum()
)

# Union-Find: nối các records thuộc cùng profile / patient
parent = list(range(len(df)))

def find_root(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def union(a, b):
    ra, rb = find_root(a), find_root(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

first_profile = {}

for i, key in enumerate(profile_key):
    if key in first_profile:
        union(i, first_profile[key])
    else:
        first_profile[key] = i

# Nếu dữ liệu có mã bệnh nhân
if "Patient_ID" in df.columns:
    first_patient = {}

    for i, patient in enumerate(df["Patient_ID"]):
        if pd.isna(patient) or str(patient).strip() == "":
            continue

        patient = str(patient).strip()

        if patient in first_patient:
            union(i, first_patient[patient])
        else:
            first_patient[patient] = i

df["_group"] = [
    f"G{find_root(i):06d}"
    for i in range(len(df))
]

print("Records:", len(df))
print("Unique groups:", df["_group"].nunique())
print("Repeated feature profiles:", n_duplicate_profiles)
print("Profiles with conflicting labels:", n_conflicting_profiles)

Records: 6058
Unique groups: 6057
Repeated feature profiles: 1
Profiles with conflicting labels: 0


In [9]:
def class_balance_error(all_y, selected_y):
    classes = ["Low", "Mid", "High"]

    p_all = np.array([
        np.mean(all_y == c) for c in classes
    ])

    p_selected = np.array([
        np.mean(selected_y == c) for c in classes
    ])

    return np.abs(p_all - p_selected).mean()


def best_group_split(data, test_fraction, random_seed):
    splitter = GroupShuffleSplit(
        n_splits=200,
        test_size=test_fraction,
        random_state=random_seed
    )

    best_score = float("inf")
    best_pair = None

    all_y = data[TARGET].to_numpy()

    for train_idx, test_idx in splitter.split(
        data,
        y=data[TARGET],
        groups=data["_group"]
    ):
        a = data.iloc[train_idx].copy()
        b = data.iloc[test_idx].copy()

        # Mỗi phía phải đủ ba lớp
        if a[TARGET].nunique() < 3:
            continue
        if b[TARGET].nunique() < 3:
            continue

        size_error = abs(
            len(b) / len(data) - test_fraction
        )

        balance_error = class_balance_error(
            all_y,
            b[TARGET].to_numpy()
        )

        score = (
            8 * size_error +
            2 * balance_error
        )

        if score < best_score:
            best_score = score
            best_pair = (a, b)

    if best_pair is None:
        raise ValueError(
            "Không tìm được group split hợp lệ."
        )

    return best_pair


# Bước 1: Tách Test 20%
development, test = best_group_split(
    df, 0.20, SEED + 1
)

# Bước 2: Lấy Train = 75% của 80% còn lại
# => 60% tổng dataset
train, remaining = best_group_split(
    development, 0.25, SEED + 2
)

# Bước 3: Chia 20% còn lại thành 10% + 10%
calibration, policy_val = best_group_split(
    remaining, 0.50, SEED + 3
)

splits = {
    "train": train,
    "calibration": calibration,
    "policy_val": policy_val,
    "test": test
}

# Kiểm tra không trùng dòng / groups
all_ids = []
group_sets = {}

for name, part in splits.items():
    all_ids.extend(part["row_id"].tolist())
    group_sets[name] = set(part["_group"])

assert len(all_ids) == len(df)
assert len(set(all_ids)) == len(df)

names = list(splits)

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        overlap = (
            group_sets[names[i]] &
            group_sets[names[j]]
        )
        assert not overlap, (
            f"Group overlap: {names[i]} / {names[j]}"
        )

print("SPLITTING PASSED!")

for name, part in splits.items():
    print(
        name,
        len(part),
        f"({len(part)/len(df):.2%})"
    )

SPLITTING PASSED!
train 3634 (59.99%)
calibration 606 (10.00%)
policy_val 606 (10.00%)
test 1212 (20.01%)


In [10]:
split_dir = OUTPUT_DIR / "splits"
split_dir.mkdir(exist_ok=True)

# Chỉ xuất các predictor, label, row_id
export_columns = ["row_id"] + FEATURES + [TARGET]

for name, part in splits.items():
    path = split_dir / f"{name}.csv"

    part[export_columns].sort_values(
        "row_id"
    ).to_csv(path, index=False)

# Xuất toàn bộ dữ liệu đã clean
df[export_columns].sort_values(
    "row_id"
).to_csv(
    OUTPUT_DIR / "model_ready.csv",
    index=False
)

# Assignment: record nào thuộc tập nào
assignments = pd.concat([
    pd.DataFrame({
        "row_id": part["row_id"],
        "split": name
    })
    for name, part in splits.items()
]).sort_values("row_id")

assignments.to_csv(
    OUTPUT_DIR / "split_assignments.csv",
    index=False
)

# Bảng phân bố lớp
summary_rows = []

for name, part in splits.items():
    counts = part[TARGET].value_counts()

    summary_rows.append({
        "Split": name,
        "Records": len(part),
        "Percentage": round(
            len(part) / len(df) * 100, 2
        ),
        "Low": int(counts.get("Low", 0)),
        "Mid": int(counts.get("Mid", 0)),
        "High": int(counts.get("High", 0))
    })

summary = pd.DataFrame(summary_rows)

summary.to_csv(
    OUTPUT_DIR / "split_summary.csv",
    index=False
)

display(summary)

print("All CSV files saved!")

,Split,Records,Percentage,Low,Mid,High
0,train,3634,59.99,1199,1225,1210
1,calibration,606,10.00,200,204,202
2,policy_val,606,10.00,200,205,201
3,test,1212,20.01,400,409,403


All CSV files saved!


In [11]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

fold_results = []

for fold, (_, val_indices) in enumerate(
    cv.split(
        train[FEATURES],
        train[TARGET],
        groups=train["_group"]
    ),
    start=1
):
    fold_results.append(
        pd.DataFrame({
            "row_id": train.iloc[val_indices]["row_id"],
            "cv_fold": fold
        })
    )

cv_assignments = pd.concat(
    fold_results,
    ignore_index=True
).sort_values("row_id")

assert cv_assignments["row_id"].is_unique
assert len(cv_assignments) == len(train)

cv_assignments.to_csv(
    OUTPUT_DIR / "train_cv_folds.csv",
    index=False
)

print("5-fold assignments saved!")
print(cv_assignments["cv_fold"].value_counts().sort_index())

5-fold assignments saved!
cv_fold
1    726
2    726
3    727
4    727
5    728
Name: count, dtype: int64


In [12]:

from pathlib import Path

print("Thư mục làm việc:", Path.cwd())
print("Thư mục output:", OUTPUT_DIR.resolve())

for file in OUTPUT_DIR.rglob("*"):
    if file.is_file():
        print(file.resolve())


Thư mục làm việc: /Users/hgianee/Downloads/Maternal health risk data 2
Thư mục output: /Users/hgianee/Downloads/Maternal health risk data 2/output
/Users/hgianee/Downloads/Maternal health risk data 2/output/train_cv_folds.csv
/Users/hgianee/Downloads/Maternal health risk data 2/output/removed_outliers_audit.csv
/Users/hgianee/Downloads/Maternal health risk data 2/output/split_assignments.csv
/Users/hgianee/Downloads/Maternal health risk data 2/output/split_summary.csv
/Users/hgianee/Downloads/Maternal health risk data 2/output/model_ready.csv
/Users/hgianee/Downloads/Maternal health risk data 2/output/splits/policy_val.csv
/Users/hgianee/Downloads/Maternal health risk data 2/output/splits/calibration.csv
/Users/hgianee/Downloads/Maternal health risk data 2/output/splits/test.csv
/Users/hgianee/Downloads/Maternal health risk data 2/output/splits/train.csv
